- PYRONEAR-2025 data-lever fine-tune | Kaggle 2x T4
  - Mục đích: tiếp tục fine-tune winner `rfdetr_large_dfire` (checkpoint_best_total.pth) trên PYRONEAR-2025-clean (data-only lever, P1 roadmap) — đo AUROC FIgLib trước/sau qua đúng harness `temporal_eval.py` sẵn có, KHÔNG dùng FIgLib để chọn epoch/checkpoint trong lúc train.
  - Kaggle Input cần 2 dataset: (1) `pyronear2025-clean` (layout `{train,val,test}/{images,labels}` + `data.yaml`, xem `filter_report.json` đi kèm để biết rule loại leakage/remap class), (2) `rfdetr-large-dfire-winner` chứa `checkpoint_best_total.pth` (dùng làm `pretrain_weights`, KHÔNG phải resume full-state)
  - Protocol: resolution GIỮ NGUYÊN 800px (khớp training gốc của winner, tránh confound resolution — eval FIgLib vẫn luôn dùng imgsz=1280 riêng biệt như mọi candidate khác), seed 20260707, effective batch 16 (2/GPU x 2GPU x accumulation 4), fp16 (T4 không có bf16), augmentation mặc định + HorizontalFlip 0.5 (khớp D-Fire/Pyro-SDIS gốc)
  - **Epoch budget = 8 (judgment call đầu tiên, CHƯA validate)** — đây là continued fine-tune từ checkpoint đã tốt (không phải from-scratch), quy mô data (32,784 train ảnh) không nhỏ; nếu Kaggle session-limit chặn giữa chừng, cell resume tự quét checkpoint dở dang và tiếp tục qua các lần chạy — không cần lo hết giờ giữa chừng
  - Sau khi train xong: convert checkpoint sang inference-ready (`.pth` best/EMA), TẢI VỀ, copy vào `artifacts/smoke_fire_detection/runs/rfdetr_large_dfire_pyronear2025_ft/`, rồi chạy `eval.py detector-cache` + `temporal_eval.py g0`/`compare-candidates` để so với winner gốc (control) — bước này làm ở server, không làm trên Kaggle
  - Early stopping: TẮT (`early_stopping=False`) — giữ đủ epoch budget đã khai báo trước, khớp policy toàn dự án (không early-stop khi đã chốt protocol)


In [ ]:
!find /kaggle/input -maxdepth 6 -type d | head -100


In [ ]:
from pathlib import Path
import shutil

SEED = 20260707
EPOCHS = 8
RESOLUTION = 800
MICRO_BATCH = 2
GRAD_ACCUM = 4
INPUT_BASE = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
RUN_NAME = 'rfdetr_large_dfire_pyronear2025_ft'
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME
SHIM_DIR = WORK_ROOT / 'pyronear2025-clean-rfdetr'

# Xoá shim cũ nếu chạy lại sau lỗi
shutil.rmtree(SHIM_DIR, ignore_errors=True)

dataset_candidates = sorted({path.parent.resolve() for path in INPUT_BASE.rglob('images')
                             if (path / 'train').is_dir() and (path / 'val').is_dir()
                             and (path.parent / 'labels' / 'train').is_dir()
                             and (path.parent / 'labels' / 'val').is_dir()})
assert len(dataset_candidates) == 1, f'Cần đúng 1 dataset pyronear2025-clean, thấy: {dataset_candidates}'
DATA_SRC = dataset_candidates[0]

checkpoint_candidates = list(INPUT_BASE.rglob('checkpoint_best_total.pth'))
assert len(checkpoint_candidates) == 1, f'Cần đúng 1 checkpoint_best_total.pth (winner rfdetr_large_dfire), thấy: {checkpoint_candidates}'
PRETRAIN_WEIGHTS = checkpoint_candidates[0]

for src, dst_name in (('train', 'train'), ('val', 'valid')):
    for sub in ('images', 'labels'):
        dst = SHIM_DIR / dst_name / sub
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.is_symlink():
            dst.symlink_to(DATA_SRC / sub / src, target_is_directory=True)

(SHIM_DIR / 'data.yaml').write_text('nc: 1\nnames:\n  0: smoke\n')

train_count = sum(1 for _ in (SHIM_DIR / 'train' / 'images').iterdir())
val_count = sum(1 for _ in (SHIM_DIR / 'valid' / 'images').iterdir())
print({'data_src': str(DATA_SRC), 'pretrain_weights': str(PRETRAIN_WEIGHTS), 'shim': str(SHIM_DIR), 'run_dir': str(RUN_DIR), 'train': train_count, 'val': val_count})


In [ ]:
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rfdetr[train]==1.8.3'], check=True)

import importlib.metadata
import torch

devices = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
capabilities = [torch.cuda.get_device_capability(index) for index in range(torch.cuda.device_count())]
cuda_version = tuple(int(part) for part in torch.version.cuda.split('.')[:2])
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'rfdetr': importlib.metadata.version('rfdetr'), 'devices': devices, 'capabilities': capabilities})
assert len(devices) == 2 and all('T4' in name for name in devices), f'Cần Accelerator GPU T4 x2, hiện có {devices}'
assert all(value >= (7, 0) for value in capabilities), f'GPU không được torch {torch.__version__} hỗ trợ: {capabilities}'
assert cuda_version >= (12, 8), f'Cần Kaggle CUDA runtime >= 12.8, hiện có {torch.version.cuda}'


In [ ]:
from collections.abc import Mapping
import json
import os
import shutil
import torch

PROTOCOL = {'schema': 1, 'framework': 'rfdetr-1.8.3', 'model': 'RFDETRLarge', 'dataset': 'pyronear2025-clean', 'base_checkpoint': str(PRETRAIN_WEIGHTS.name), 'resolution': RESOLUTION, 'epochs': EPOCHS, 'seed': SEED, 'micro_batch': MICRO_BATCH, 'grad_accum': GRAD_ACCUM, 'devices': 2, 'strategy': 'ddp_notebook', 'amp_dtype': 'fp16'}
protocol = RUN_DIR / 'resume_protocol.json'
existed = RUN_DIR.exists()

def valid_checkpoint(path, source):
    state = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(state, Mapping) or not isinstance(state.get('epoch'), int) or not isinstance(state.get('global_step'), int):
        raise ValueError('thiếu epoch/global_step')
    if not isinstance(state.get('state_dict'), Mapping) or not state['state_dict'] or not state.get('optimizer_states') or not state.get('lr_schedulers'):
        raise ValueError('không full-state')
    q = protocol if source == 'working' else next((x / 'resume_protocol.json' for x in (path.parent, *path.parents) if (x / 'resume_protocol.json').is_file()), None)
    if q is None or json.loads(q.read_text()) != PROTOCOL:
        raise ValueError('protocol không khớp')
    return {'path': path, 'source': source, 'epoch': state['epoch'], 'step': state['global_step']}

def scan(paths, source):
    accepted = []
    for path in paths:
        try:
            accepted.append(valid_checkpoint(path, source))
        except Exception as error:
            print('reject', source, path, error)
    return accepted

if existed and not protocol.is_file():
    raise RuntimeError('working thiếu protocol')

working_checkpoints = scan(RUN_DIR.glob('checkpoint_*.ckpt'), 'working')
if existed and not working_checkpoints:
    raise RuntimeError('working không có checkpoint hợp lệ')
input_checkpoints = scan(INPUT_BASE.rglob('checkpoint_*.ckpt'), 'input')
chosen = max(working_checkpoints + input_checkpoints, key=lambda item: (item['epoch'], item['step'], item['source'] == 'working'), default=None)

if chosen and chosen['source'] == 'input':
    destination = RUN_DIR / 'input_epoch_{:03d}_step_{:09d}.ckpt'.format(chosen['epoch'], chosen['step'])
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix('.tmp')
    shutil.copyfile(chosen['path'], temporary)
    os.replace(temporary, destination)
    resume_path = destination
elif chosen:
    resume_path = chosen['path']
else:
    resume_path = None

if not chosen and not existed:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    protocol.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True), encoding='utf-8')

run_complete = bool(chosen and chosen['epoch'] >= EPOCHS - 1)
print({'resume_source': chosen['source'] if chosen else 'fresh (from pretrain_weights)', 'resume_path': str(resume_path) if resume_path else None, 'complete': run_complete})


In [ ]:
from rfdetr import RFDETRLarge

if run_complete:
    print(f'Không train lại: checkpoint đã hoàn thành epoch {EPOCHS}.')
else:
    model = RFDETRLarge(resolution=RESOLUTION, pretrain_weights=str(PRETRAIN_WEIGHTS))
    model.train(
        dataset_dir=str(SHIM_DIR), dataset_file='yolo', output_dir=str(RUN_DIR),
        epochs=EPOCHS, batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM,
        accelerator='gpu', devices=2, strategy='ddp_notebook', amp_dtype='fp16', num_workers=2,
        checkpoint_interval=1, seed=SEED, early_stopping=False, tensorboard=False,
        multi_scale=False, aug_config={'HorizontalFlip': {'p': 0.5}},
        warmup_epochs=0.0, lr_scheduler='cosine', lr_min_factor=0.01,
        resume=str(resume_path) if resume_path else None,
    )


In [ ]:
checkpoints = sorted(RUN_DIR.glob('checkpoint_*.ckpt'))
weights = sorted(RUN_DIR.glob('*.pth'))
for path in checkpoints + weights:
    print(path, path.stat().st_size)
assert checkpoints
assert weights
